# 06 - Poisson Equation

Solve $-u_{xx} = \pi^2 \sin(\pi x)$

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, sys
sys.path.insert(0, '../src')
from pinns import MLP, set_seed
from pinns.utils.derivatives import gradient
set_seed(42)
model = MLP(input_dim=1, output_dim=1, hidden_dims=[32,32,32], activation='tanh')

In [ ]:
def pde_loss(model, x):
    x = x.requires_grad_(True)
    u = model(x)
    u_x = gradient(u, x)
    u_xx = gradient(u_x, x)
    return torch.mean((-u_xx - np.pi**2 * torch.sin(np.pi*x))**2)
def bc_loss(model):
    return model(torch.zeros(1,1))**2 + model(torch.ones(1,1))**2
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(1500):
    opt.zero_grad()
    loss = pde_loss(model, torch.rand(100,1)) + 100*bc_loss(model)
    loss.backward(); opt.step()
    if (epoch+1) % 500 == 0: print(f'Epoch {epoch+1}: {loss.item():.4e}')

In [ ]:
x = torch.linspace(0, 1, 100).reshape(-1, 1)
with torch.no_grad(): u_pred = model(x)
u_exact = torch.sin(np.pi*x)
plt.figure(figsize=(10,5))
plt.plot(x.numpy(), u_exact.numpy(), 'b-', lw=2, label='Exact')
plt.plot(x.numpy(), u_pred.numpy(), 'r--', lw=2, label='PINN')
plt.legend(); plt.grid(True, alpha=0.3)
plt.xlabel('x'); plt.ylabel('u'); plt.title('Poisson Equation')
plt.tight_layout(); plt.show()
print(f'MAE: {torch.abs(u_pred - u_exact).mean():.6e}')